<a href="https://colab.research.google.com/github/joseguzman1337/aws-financial-ai-agent/blob/main/invocation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end passwordless AWS authentication, live agent invocation, and observability trace extraction for recruiters. It uses AWS SigV4 + IAM credentials from the Colab runtime and retrieves observability secrets from AWS SSM securely.


In [ ]:
# Install dependencies silently
!pip install boto3 requests langfuse -q
!sudo apt-get update -y -q > /dev/null && sudo apt-get install neofetch -y -q > /dev/null
!neofetch

import json
import os
import time
import urllib.parse
import uuid

import boto3
import requests
from botocore import UNSIGNED
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.config import Config
from botocore.session import Session

CFG = {
    "region": "us-east-1",
    "agent_arn": "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-hvRgckAqaW",
    "identity_pool_id": "us-east-1:c7680c24-fe96-4358-b305-6f43de1ca6c8",
    "unauth_role_arn": "arn:aws:iam::162187491349:role/cognito_unauthenticated_role",
    "credential_refresh_seconds": 45 * 60,
}

PARAMS = {
    "langfuse_pk": "/financial-ai/langfuse/public-key",
    "langfuse_sk": "/financial-ai/langfuse/secret-key",
    "langfuse_base_url": "/financial-ai/langfuse/base-url",
}

AGENTCORE_URL = (
    f"https://bedrock-agentcore.{CFG['region']}.amazonaws.com/runtimes/"
    f"{urllib.parse.quote(CFG['agent_arn'], safe='')}/invocations"
)
MASTER_SESSION_ID = str(uuid.uuid4())
LAST_CREDENTIAL_REFRESH_EPOCH = 0

sts = boto3.client("sts", region_name=CFG["region"])
ssm = boto3.client("ssm", region_name=CFG["region"])


def refresh_clients():
    global sts, ssm
    sts = boto3.client("sts", region_name=CFG["region"])
    ssm = boto3.client("ssm", region_name=CFG["region"])


def bootstrap_guest_aws_credentials() -> bool:
    """Passwordless bootstrap via Cognito Identity Pool classic flow."""
    global LAST_CREDENTIAL_REFRESH_EPOCH
    try:
        idc = boto3.client(
            "cognito-identity",
            region_name=CFG["region"],
            config=Config(signature_version=UNSIGNED),
        )
        identity_id = idc.get_id(IdentityPoolId=CFG["identity_pool_id"])["IdentityId"]
        token = idc.get_open_id_token(IdentityId=identity_id)["Token"]
        creds = boto3.client("sts", region_name=CFG["region"]).assume_role_with_web_identity(
            RoleArn=CFG["unauth_role_arn"],
            RoleSessionName="NotebookGuestSession",
            WebIdentityToken=token,
        )["Credentials"]

        os.environ["AWS_ACCESS_KEY_ID"] = creds["AccessKeyId"]
        os.environ["AWS_SECRET_ACCESS_KEY"] = creds["SecretAccessKey"]
        os.environ["AWS_SESSION_TOKEN"] = creds["SessionToken"]
        refresh_clients()
        LAST_CREDENTIAL_REFRESH_EPOCH = int(time.time())
        return True
    except Exception as e:
        print(f"Bootstrap guest failed: {e}")
        return False


def ssm_get(name: str) -> str:
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except Exception as e:
        if "ExpiredToken" in str(e) or "ExpiredTokenException" in str(e):
            if bootstrap_guest_aws_credentials():
                return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
        raise


def ensure_fresh_credentials(force: bool = False) -> bool:
    """Refresh credentials proactively and on ExpiredToken."""
    global LAST_CREDENTIAL_REFRESH_EPOCH
    now = int(time.time())
    refresh_age = now - int(LAST_CREDENTIAL_REFRESH_EPOCH or 0)
    if force or LAST_CREDENTIAL_REFRESH_EPOCH == 0 or refresh_age >= CFG["credential_refresh_seconds"]:
        return bootstrap_guest_aws_credentials()
    try:
        sts.get_caller_identity()
        return True
    except Exception as e:
        if "ExpiredToken" in str(e) or "ExpiredTokenException" in str(e):
            print("Token expired. Refreshing guest credentials...")
            return bootstrap_guest_aws_credentials()
        raise


def _sigv4_headers(payload: bytes) -> dict:
    req = AWSRequest(
        method="POST",
        url=AGENTCORE_URL,
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Accept": "text/event-stream",
            "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": MASTER_SESSION_ID,
        },
    )
    SigV4Auth(
        Session().get_credentials().get_frozen_credentials(),
        "bedrock-agentcore",
        CFG["region"],
    ).add_auth(req)
    return dict(req.prepare().headers)


def _event_text(event) -> str:
    if isinstance(event, str): return event
    if isinstance(event, list): return "".join(_event_text(p) for p in event)
    if isinstance(event, dict): return str(event.get("text") or _event_text(event.get("content", "")))
    return str(event)


def query_financial_agent(prompt: str):
    ensure_fresh_credentials()
    payload = json.dumps({"prompt": prompt}).encode("utf-8")
    print(f"\n--- Query: {prompt} ---")
    resp = requests.post(AGENTCORE_URL, headers=_sigv4_headers(payload), data=payload, stream=True, timeout=120)
    if resp.status_code != 200:
        print(f"Error: {resp.status_code} {resp.text[:350]}")
        return
    for line in resp.iter_lines():
        if line:
            decoded = line.decode("utf-8")
            if decoded.startswith("data: "):
                data = json.loads(decoded[6:])
                if "event" in data: print(_event_text(data["event"]), end="")
    print()





### 1. Verify AWS Identity
Confirm the notebook runtime has AWS credentials and can call AWS APIs.


In [ ]:
print("Refreshing identity...")
try:
    if not bootstrap_guest_aws_credentials():
        raise RuntimeError("Guest bootstrap failed")
    identity = sts.get_caller_identity()
    print(f"Success: Passwordless Cognito guest identity {identity['Arn']}")
except Exception as e:
    print(f"Critical Identity Error: {e}")



### 2. SigV4 Auth Mode
No Cognito username/password is required. Agent calls are signed directly with AWS SigV4.


In [ ]:
print("No Cognito password flow required in notebook.")
print("Mode: Passwordless Cognito classic-flow + SigV4.")



### 3. Invoke the Agentcore Streaming Endpoint

In [54]:
print(f"Agent URL configured: {AGENTCORE_URL}")
print(f"Session ID: {MASTER_SESSION_ID}")


Agent URL configured: https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A162187491349%3Aruntime%2FFinancial_Analyst_Agent-hvRgckAqaW/invocations
Session ID: bc0fe8ce-0b14-41a0-abec-e25fd36b9495


### 4. Execute Required Financial Queries

In [ ]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]
for q in queries:
    ensure_fresh_credentials()
    query_financial_agent(q)



### 5. Observability Verification
Fetch Langfuse traces using keys from AWS SSM and verify API connectivity. Session traces may appear with delay.


In [ ]:
try:
    ensure_fresh_credentials()
    pk = ssm_get(PARAMS["langfuse_pk"])
    sk = ssm_get(PARAMS["langfuse_sk"])
    base = ssm_get(PARAMS["langfuse_base_url"]).rstrip("/")

    who = sts.get_caller_identity()
    print(f"Observability identity: {who['Arn']}")
    print(f"Success: retrieved Langfuse keys (PK: {pk[:7]}...)")

    auth = requests.get(f"{base}/api/public/projects", auth=(pk, sk), timeout=30)
    if auth.status_code == 200:
        print("Success: Langfuse API authentication verified.")
    else:
        raise RuntimeError(f"Langfuse API auth failed: HTTP {auth.status_code}")

    print(f"Searching traces for Session ID: {MASTER_SESSION_ID}")
    found = False
    for wait_s in [5, 10, 20, 30]:
        print(f"Waiting {wait_s}s for trace propagation...")
        time.sleep(wait_s)
        tr = requests.get(
            f"{base}/api/public/traces?sessionId={MASTER_SESSION_ID}",
            auth=(pk, sk),
            timeout=30,
        )
        traces = tr.json().get("data", []) if tr.status_code == 200 else []
        if traces:
            print("\n--- Langfuse Session Trace ---")
            print(json.dumps(traces[0], indent=2)[:2000])
            found = True
            break

    if not found:
        print("No session traces yet. Emitting a fallback verification trace...")
        from langfuse import get_client

        os.environ["LANGFUSE_PUBLIC_KEY"] = pk
        os.environ["LANGFUSE_SECRET_KEY"] = sk
        os.environ["LANGFUSE_BASE_URL"] = base

        lf = get_client()
        with lf.start_as_current_observation(
            as_type="span",
            name="notebook-observability-check",
            input={"session_id": MASTER_SESSION_ID},
        ) as span:
            span.update(output={"status": "ok"})
        lf.flush()

        time.sleep(3)
        recent = requests.get(f"{base}/api/public/traces?limit=5", auth=(pk, sk), timeout=30)
        data = recent.json().get("data", []) if recent.status_code == 200 else []
        if data:
            print("Fallback trace emitted successfully. Latest trace:")
            print(json.dumps(data[0], indent=2)[:2000])
        else:
            print("Fallback trace not visible yet, but API connectivity is healthy.")

except Exception as e:
    if "ExpiredToken" in str(e) or "ExpiredTokenException" in str(e):
        print("Observability token expired. Refreshing credentials and retrying once...")
        if bootstrap_guest_aws_credentials():
            try:
                pk = ssm_get(PARAMS["langfuse_pk"])
                sk = ssm_get(PARAMS["langfuse_sk"])
                base = ssm_get(PARAMS["langfuse_base_url"]).rstrip("/")
                print("Retry succeeded after token refresh.")
            except Exception as retry_err:
                print(f"Observability retry failed: {retry_err}")
        else:
            print("Credential refresh failed.")
    else:
        print(f"Observability verification failed: {e}")


